# Dataset preparation

In [1]:
# imports
from typing import Literal
from datasets import Dataset, DatasetDict, load_dataset, enable_progress_bars
from huggingface_hub.utils import enable_progress_bars as enable_hub_progress_bars

enable_progress_bars()
enable_hub_progress_bars()
import random
from collections import Counter, defaultdict
import numpy as np

import torch
from torch.utils.data import Dataset, DataLoader

from pathlib import Path
import pickle

## Load the Yambda dataset

In [2]:
# from the Hugging Face dataset


class YambdaDataset:
    INTERACTIONS = frozenset(
        ["likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"]
    )

    def __init__(
        self,
        dataset_type: Literal["flat", "sequential"] = "flat",
        dataset_size: Literal["50m", "500m", "5b"] = "50m",
    ):
        assert dataset_type in {"flat", "sequential"}
        assert dataset_size in {"50m", "500m", "5b"}
        self.dataset_type = dataset_type
        self.dataset_size = dataset_size

    def interaction(
        self,
        event_type: Literal[
            "likes", "listens", "multi_event", "dislikes", "unlikes", "undislikes"
        ],
    ) -> Dataset:
        assert event_type in YambdaDataset.INTERACTIONS
        return self._download(f"{self.dataset_type}/{self.dataset_size}", event_type)

    def audio_embeddings(self) -> Dataset:
        return self._download("", "embeddings")

    def album_item_mapping(self) -> Dataset:
        return self._download("", "album_item_mapping")

    def artist_item_mapping(self) -> Dataset:
        return self._download("", "artist_item_mapping")

    @staticmethod
    def _download(data_dir: str, file: str) -> Dataset:
        data = load_dataset(
            "yandex/yambda", data_dir=data_dir, data_files=f"{file}.parquet"
        )
        # Returns DatasetDict; extracting the only split
        assert isinstance(data, DatasetDict)
        return data["train"]


dataset = YambdaDataset("flat", "50m")

In [3]:
# for the baseline model with basic next token prediction we will use the listens history

listens = dataset.interaction("listens")

In [4]:
print(listens)
print("Number of examples:", len(listens))

Dataset({
    features: ['uid', 'timestamp', 'item_id', 'is_organic', 'played_ratio_pct', 'track_length_seconds'],
    num_rows: 46467212
})
Number of examples: 46467212


## Sample a subset of users (so my laptop doesn't die)

In [5]:
user_ids = listens.unique("uid")
print(f"We have {len(user_ids)} unique users")

We have 9238 unique users


In [6]:
seed = random.Random(3529)

n_users = min(50, len(user_ids))
user_ids_subset = set(seed.sample(user_ids, n_users))

In [7]:
# get events from selected users
events = listens.filter(
    lambda row: row["uid"] in user_ids_subset,
)
print(
    f"The number of selected users is {len(user_ids_subset)} and the total number of events is {len(events)}"
)

The number of selected users is 50 and the total number of events is 278196


## Create splits (and make sure to avoid temporal leakage)

In [8]:
timestamps = np.asarray(events["timestamp"], dtype=np.int64)

t_train = np.quantile(timestamps, 0.7)
t_validate = np.quantile(timestamps, 0.85)

print("Training boundary:", t_train)
print("Testing boundary:", t_validate)

Training boundary: 20618992.5
Testing boundary: 23603395.0


In [9]:
# split the data

train = events.filter(
    lambda row: row["timestamp"] < t_train,
    desc="Creating training split",
)

validate = events.filter(
    lambda row: t_train <= row["timestamp"] < t_validate,
    desc="Creating validation split",
)

test = events.filter(
    lambda row: row["timestamp"] >= t_validate,
    desc="Creating testing split",
)

print(
    f"We have {len(train)} training events, {len(validate)} validation events, and {len(test)} testing events"
)

splits = {
    "train": train,
    "validation": validate,
    "test": test,
}

We have 194737 training events, 41728 validation events, and 41731 testing events


In [10]:
user_wise_sequences = defaultdict(
    lambda: {
        "train": [],
        "validation": [],
        "test": [],
    }
)

for split_name, split_data in splits.items():
    for row in split_data:
        uid = row["uid"]
        item_id = row["item_id"]
        user_wise_sequences[uid][split_name].append(item_id)

user_sequences = dict(user_wise_sequences)

In [11]:
# example
print(user_sequences[next(iter(user_sequences))])

{'train': [7094195, 117174, 9143560, 121773, 1669744, 6699637, 1860416, 6690987, 7421463, 4409793, 500962, 7735065, 3987314, 3095165, 9267109, 7365232, 7451308, 8240867, 7404746, 6666341, 2397653, 2791756, 1361026, 2146480, 7181370, 5486396, 7343381, 193619, 9228104, 8081588, 3368872, 7047232, 8285928, 3162353, 8963417, 1532204, 6758971, 1532204, 9349529, 2371247, 652052, 9116473, 2524742, 4738870, 7398241, 8793356, 4904790, 6182516, 3072368, 3361906, 4038308, 841550, 2026411, 1315922, 4446418, 374835, 2541842, 8698897, 1509749, 8144152, 3977539, 8579146, 6471327, 7131766, 7081282, 4470480, 7915747, 8428805, 7597551, 7928234, 8246535, 4710357, 4779028, 7013349, 7074794, 5640427, 5313316, 3527891, 6715825, 5165597, 6835614, 6368299, 2344603, 7185409, 3866116, 4034702, 3427453, 3227801, 8519800, 9279277, 8035618, 7702134, 3492745, 758005, 161903, 6296932, 5952627, 2929378, 2073854, 3185743, 5282314, 2100087, 4827175, 7342647, 3936230, 6095278, 8461721, 904479, 8294825, 5009838, 6277714, 

## Vocabulary mapping

In [12]:
item_counts = Counter(
    item_id for sequences in user_sequences.values() for item_id in sequences["train"]
)

# for the sake of time and computing resources, we will start with a preset number of most common items :)
N_FREQUENT = 1000

most_common_items = item_counts.most_common(N_FREQUENT)

# mappings
item_to_token = {
    item_id: token_id for token_id, (item_id, _) in enumerate(most_common_items)
}

token_to_item = {
    token_id: item_id for item_id, token_id in item_to_token.items()
}

print("Vocabulary size:", len(item_to_token))
print("Example of the item-token mappings:", list(item_to_token.items())[:10])

Vocabulary size: 1000
Example of the item-token mappings: [(5862961, 0), (9010147, 1), (2859641, 2), (5734320, 3), (7327785, 4), (5773949, 5), (2651314, 6), (2117059, 7), (2383009, 8), (3033749, 9)]


In [13]:
# work around rare items without throwing them out of consecutive sequences
def extract_contiguous_seqs(item_ids, item_to_token):
    runs = []
    current_run = []

    for item_id in item_ids:
        if item_id in item_to_token:
            current_run.append(item_to_token[item_id])
        else:
            if current_run:
                runs.append(current_run)
                current_run = []

    if current_run:
        runs.append(current_run)

    return runs

In [14]:
train_sequences = []

for sequences in user_sequences.values():
    runs = extract_contiguous_seqs(
        sequences["train"],
        item_to_token,
    )
    train_sequences.extend(runs)

train_sequences = [
    seq for seq in train_sequences
    if len(seq) >= 2
]

print("Training sequences:", len(train_sequences))
print("Example of the encoded sequence:", train_sequences[239])

Training sequences: 7920
Example of the encoded sequence: [892, 612, 537, 475]


## Create a PyTorch dataset & DataLoaders

In [15]:
class NextItemDataset(Dataset):
    def __init__(self, sequences, max_length=32):
        self.sequences = []
        for seq in sequences:
            start = 0
            while start < len(seq) - 1:
                chunk = seq[start:start + max_length]
                if len(chunk) >= 2:
                    self.sequences.append(chunk)
                # overlap
                start += max_length - 1

    def __len__(self):
        return len(self.sequences)

    def __getitem__(self, idx):
        return torch.tensor(
            self.sequences[idx],
            dtype=torch.long,
        )

In [16]:
# now we will pad sequences so all of them have equel lengths

PAD_ID = len(item_to_token)
MAX_LENGTH = 32
BATCH_SIZE = 8

def collate_batch(seqs):
    max_len = max(len(seq) for seq in seqs)

    input_ids = torch.full((len(seqs), max_len), PAD_ID, dtype=torch.long,
    )

    attention_mask = torch.zeros((len(seqs), max_len), dtype=torch.long,
    )

    labels = torch.full(
        (len(seqs), max_len), -100, dtype=torch.long,
    )

    for i, seq in enumerate(seqs):
        n = len(seq)
        input_ids[i, :n] = seq
        attention_mask[i, :n] = 1
        labels[i, :n] = seq # just the same tokens for the causal lang modeling (there will be automatuc shifts)

    return {
        "input_ids": input_ids, # those are tokens
        "attention_mask": attention_mask, # zeros for paddings
        "labels": labels, # also the same tokens, -100 for paddings 
    }

train_dataset = NextItemDataset(
    train_sequences,
    max_length=MAX_LENGTH,
)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_batch,
)
print(f"We have {len(train_dataset)} chunks for training")

We have 8191 chunks for training


In [17]:
# add a mapping for padding
item_to_token[1000] = -100
print(len(item_to_token))

1001


In [18]:
batch = next(iter(train_loader))

for key, value in batch.items():
    print(key, value.shape, value.dtype)

print("First input_ids:", batch["input_ids"][0])
print("First labels:", batch["labels"][0])
print("First mask:", batch["attention_mask"][0])

input_ids torch.Size([8, 5]) torch.int64
attention_mask torch.Size([8, 5]) torch.int64
labels torch.Size([8, 5]) torch.int64
First input_ids: tensor([ 386,  371, 1000, 1000, 1000])
First labels: tensor([ 386,  371, -100, -100, -100])
First mask: tensor([1, 1, 0, 0, 0])


## Save data

In [19]:
output_dir = Path("data")
output_dir.mkdir(exist_ok=True)

artifacts = {
    "user_sequences": user_sequences,
    "item_to_token": item_to_token,
    "token_to_item": token_to_item,
    "train_sequences": train_sequences,
    "t_train": t_train,
    "t_validate": t_validate,
}

with open(output_dir / "artifacts.pkl", "wb") as f:
    pickle.dump(artifacts, f)

print("Artifacts saved on disk")

Artifacts saved on disk
